# PPAP Quality Review Agent — Colab Demo

**Do not paste this `.ipynb` file into a code cell.** Open it as a notebook:

- [Open this notebook in Google Colab](https://colab.research.google.com/github/rockyforever8-sys/Agentic-MDS/blob/cursor/ppap-quality-agent-17d5/PPAP_Colab_Start_Here.ipynb)
- Fallback: [prototype branch notebook](https://colab.research.google.com/github/rockyforever8-sys/Agentic-MDS/blob/cursor/ppap-langgraph-prototype-17d5/PPAP_Colab_Start_Here.ipynb)

Runtime → **Run all**. No API keys required.

Sources are cloned from **Agentic-MDS** (the standalone PPAP branch). The empty `Agentic-PPAP` repo is not used.

In [ ]:
# Cell 1 — clone working branch + install (run this first)
import os, pathlib, shutil, subprocess, sys

ROOT = pathlib.Path('/content/ppap_agent_repo')
CANDIDATES = [
    ('https://github.com/rockyforever8-sys/Agentic-MDS.git', 'cursor/ppap-quality-agent-17d5'),
    ('https://github.com/rockyforever8-sys/Agentic-MDS.git', 'cursor/ppap-langgraph-prototype-17d5'),
]

def _ok(path):
    return (path / 'ppap_agent' / '__init__.py').exists()

cloned = None
if _ok(ROOT):
    cloned = ROOT
else:
    for repo, ref in CANDIDATES:
        try:
            if ROOT.exists():
                shutil.rmtree(ROOT)
            print(f'Cloning {repo} @{ref} ...')
            subprocess.check_call(['git', 'clone', '--depth', '1', '--branch', ref, repo, str(ROOT)])
            if _ok(ROOT):
                cloned = ROOT
                print(f'Ready: {repo} @{ref}')
                break
        except subprocess.CalledProcessError as exc:
            print(f'Skip {ref}: {exc}')
            if ROOT.exists():
                shutil.rmtree(ROOT, ignore_errors=True)
if cloned is None:
    raise RuntimeError('Could not clone PPAP agent. Check git access to rockyforever8-sys/Agentic-MDS.')

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
%pip install -q langgraph langchain-core rich
print('Package path:', ROOT)
print('ppap_agent files:', sorted(p.name for p in (ROOT / 'ppap_agent').iterdir() if p.suffix == '.py'))

In [ ]:
# Cell 2 — seed synthetic database and show inbox
import os
from ppap_agent.database.seed import seed_database
from ppap_agent.database.db import list_pending_ppaps

DB = ROOT / 'ppap_agent' / 'data' / 'ppap_synthetic.db'
summary = seed_database(DB)
os.environ['PPAP_DB_PATH'] = str(DB)

print(f'Seeded: {summary["ppap_submissions"]} PPAPs, {summary["aiag_rules"]} AIAG rules')
print(f'\nInbox ({len(list_pending_ppaps())} pending):')
for p in list_pending_ppaps():
    print(f'  {p["id"]}  {p["part_number"]:20s}  {p["supplier_name"]}')

## Animated Single PPAP Review

Change `PPAP_ID` to try other cases: `PPAP-2026-001` accept, `PPAP-2026-002` hold, `PPAP-2026-003` reject.

In [ ]:
# Cell 3 — animated LangGraph review
import time
from IPython.display import HTML, clear_output, display
from ppap_agent.visualization import render_graph_html, stream_ppap_review

PPAP_ID = 'PPAP-2026-003'
DELAY = 0.45

final_state = {}
for step in stream_ppap_review(PPAP_ID):
    clear_output(wait=True)
    display(HTML(render_graph_html(
        active_nodes=step['active_nodes'],
        completed_nodes=step['completed_nodes'],
        ppap_id=PPAP_ID,
        decision=step.get('state', {}).get('decision') if step.get('done') else None,
        risk_band=step.get('state', {}).get('risk_band') if step.get('done') else None,
    )))
    print(step['message'])
    final_state = step.get('state', final_state)
    if not step.get('done'):
        time.sleep(DELAY)

d = final_state.get('decision', '?')
print(f'\nDECISION: {d.upper()}  |  Risk: {final_state.get("risk_band")} ({final_state.get("risk_score", 0):.0f}/100)')
for r in final_state.get('decision_reasons', []):
    print(f'  • {r}')
print('\nSupplier:', final_state.get('supplier_notification', ''))

## Batch Supervisor Graph

In [ ]:
# Cell 4 — process all pending PPAPs
from ppap_agent.agents.batch_graph import run_batch_review

seed_database(DB)
result = run_batch_review(max_reviews=8)
s = result['batch_summary']
print(f'Batch complete: {s["reviews_completed"]} reviews')
print(f'  Accepted: {s["accepted"]}  |  Rejected: {s["rejected"]}  |  On Hold: {s["on_hold"]}')
print(f'  Auto-accept rate: {s.get("auto_accept_rate", 0)}%\n')
for c in result['completed']:
    icon = {'accept': '✅', 'reject': '❌', 'hold': '⏸️'}.get(c['decision'], '?')
    print(f'  {icon} {c["ppap_id"]}  {c["part_number"]:20s}  {c["decision"].upper():6s}  risk={c["risk_score"]:.0f}')

## Decision matrix (all 8 scenarios)

In [ ]:
# Cell 5 — decision matrix
from ppap_agent.agents.graph import run_ppap_review

scenarios = [
    'PPAP-2026-001', 'PPAP-2026-002', 'PPAP-2026-003', 'PPAP-2026-004',
    'PPAP-2026-005', 'PPAP-2026-006', 'PPAP-2026-007', 'PPAP-2026-008',
]
print(f'{"PPAP ID":<16} {"Part":<20} {"Decision":<8} {"Risk":<6} {"Score":>5} {"Findings":>8}')
print('-' * 70)
for pid in scenarios:
    seed_database(DB)
    r = run_ppap_review(pid)
    icon = {'accept': '✅', 'reject': '❌', 'hold': '⏸️'}.get(r['decision'], '?')
    print(f'{pid:<16} {r.get("part_number",""):<20} {icon} {r["decision"]:<6} {r.get("risk_band",""):<6} {r.get("risk_score",0):5.0f} {len(r.get("all_findings",[])):>8}')